# test the trajopt method

In [1]:
from robot import Robot, MotomanRobot
from planning_scene import PlanningScene
from geometric_trajopt_ipopt import PoseTrajOpt
import numpy as np
import scipy as sp
import transformations as tf


In [2]:
# test the robot with the scene
# add environment collisions
pcd_total = []
# shelf-bottom
num_points = 1000
position = np.array([0.85, 0, 0.5])
half_size = np.array([0.175, 0.5, 0.5])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-top
num_points = 1000
position = np.array([0.85, 0, 1.42])
half_size = np.array([0.175, 0.5, 0.025])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-left
num_points = 1000
position = np.array([0.85, -0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-right
num_points = 1000
position = np.array([0.85, 0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-back
num_points = 1000
position = np.array([1.0, 0, 1.2])
half_size = np.array([0.025, 0.5, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
pcd_total = np.concatenate(pcd_total, axis=0)


torso_b1 = ["torso_joint_b1"]
left = [
    "arm_left_joint_1_s",
    "arm_left_joint_2_l",
    "arm_left_joint_3_e",
    "arm_left_joint_4_u",
    "arm_left_joint_5_r",
    "arm_left_joint_6_b",
    "arm_left_joint_7_t",
]
right = [
    "arm_right_joint_1_s",
    "arm_right_joint_2_l",
    "arm_right_joint_3_e",
    "arm_right_joint_4_u",
    "arm_right_joint_5_r",
    "arm_right_joint_6_b",
    "arm_right_joint_7_t",
]
robot_joint_names = right#torso_b1 + left + right
robot = MotomanRobot(selected_joint_names=robot_joint_names)
# scene = PlanningScene(robot, scene_pcd=pcd_total)
scene = PlanningScene(robot, scene_pcd=None)
scene.update_scene_pcd(pcd_total)


In [3]:
q_start = np.array([0.0, 0, 0, 0, 0, 0, 0])
robot.set_selected_joint_values(q_start)
scene.visualize()

[TriangleMesh with 194 points and 384 triangles.,
 TriangleMesh with 579 points and 1154 triangles.,
 TriangleMesh with 732 points and 1460 triangles.,
 TriangleMesh with 803 points and 1590 triangles.,
 TriangleMesh with 451 points and 898 triangles.,
 TriangleMesh with 694 points and 1384 triangles.,
 TriangleMesh with 527 points and 1050 triangles.,
 TriangleMesh with 464 points and 924 triangles.,
 TriangleMesh with 225 points and 446 triangles.,
 TriangleMesh with 732 points and 1460 triangles.,
 TriangleMesh with 803 points and 1590 triangles.,
 TriangleMesh with 451 points and 898 triangles.,
 TriangleMesh with 694 points and 1384 triangles.,
 TriangleMesh with 527 points and 1050 triangles.,
 TriangleMesh with 464 points and 924 triangles.,
 TriangleMesh with 225 points and 446 triangles.,
 TriangleMesh with 10905 points and 21834 triangles.,
 TriangleMesh with 17156 points and 34248 triangles.,
 TriangleMesh with 672 points and 1340 triangles.,
 TriangleMesh with 888 points an

In [4]:
# pos = np.array([0.9096643361780983, -0.22, 1.2246445275392399])
# quat = np.array([0.99997882565292, 0.0020695812292800746, -0.005795312726922581, 0.0021164663332217232])
# pose = tf.quaternion_matrix(quat)
# pose[:3,3] = pos

# * make sure the sampled joint is collision free
while True:
    # * generate the start pose by randomly sampling joint angles
    q_target = np.random.uniform(robot.selected_joint_limits[:,0], robot.selected_joint_limits[:,1])
    print('q_target:', q_target)
    # * visualize the start pose
    robot.set_selected_joint_values(q_target)
    # * check if the start pose is collision free
    col_list = scene.compute_collision_total()
    scene.visualize()
    # check for collision
    for link1, link2, obj1, obj2, col_result in col_list:
        print('collision:')
        print('link1:', link1)
        print('link2:', link2)
        print('result: ', col_result.isCollision())
        print('distance: ', -col_result.getContact(0).penetration_depth)
    if len(col_list) == 0:
        break
robot.set_selected_joint_values(q_target)
scene.visualize();

q_target: [ 0.84718928  1.71018509  1.98167181 -0.02109226 -0.67383174 -0.1148232
  1.04008167]
collision:
link1: base_mount
link2: scene
result:  True
distance:  0.0009311988298431158
collision:
link1: robotiq_2f85_base
link2: scene
result:  True
distance:  0.0009468099909654301
collision:
link1: right_driver
link2: scene
result:  True
distance:  0.0007297638167177462
collision:
link1: right_coupler
link2: scene
result:  True
distance:  -0.0013848056571425218
q_target: [ 3.03445722 -0.60460334  2.60828126  1.65514687  0.60360712 -0.49928754
  1.94854088]


In [5]:
# check fcl-obj
for link1, link2, obj1, obj2, collision_result in col_list:
    print('link1: ', link1)
    print('link2: ', link2)
    if link2 == 'scene':
        continue
    fcl_obj1 = robot.robot_link_name_to_fcl_objs[link1][obj1]
    fcl_obj2 = robot.robot_link_name_to_fcl_objs[link2][obj2]
    print('fcl_obj1 translation: ')
    print(fcl_obj1.getTranslation())
    print('fcl_obj1 rotation: ')
    print(fcl_obj1.getRotation())
    pose = robot.robot_link_name_to_transform[link1]
    geom_pose = robot.robot_link_name_to_geoms[link1][obj1]['pose']
    pose = pose @ geom_pose
    print('stored transform: ', pose)


In [6]:
# obtain the goal pose
target_link = "arm_right_link_7_t"
robot.set_selected_joint_values(q_target)
target_pose = robot.get_link_pose(target_link)
print("target_pose: ", target_pose)

target_pose:  [[ 0.82190608  0.15507341 -0.54810823 -0.03182102]
 [-0.44189742 -0.43359766 -0.78531506 -0.83354438]
 [-0.35943994  0.88766283 -0.28785001  0.96692264]
 [ 0.          0.          0.          1.        ]]


In [7]:
distance = np.linalg.norm(q_target-q_start)
# n_waypoints = int(np.ceil(distance / (2*np.pi/180)))
n_waypoints = 10
print('distsance: ', distance)
print('n_waypoints: ', n_waypoints)
dist = distance / 5
solver = PoseTrajOpt(robot, scene, start_q=q_start, target_link=target_link, target_pose=target_pose, n_waypoints=n_waypoints, max_dist=dist)

distsance:  4.850418465560178
n_waypoints:  10
lb shape:  (70,)
lb: 
[-3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159
 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986
 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706
 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619
 -3.14159 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159
 -1.91986 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986
 -3.14159 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159
 -3.14159 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159 -3.14159
 -1.91986 -2.96706 -2.35619 -3.14159 -1.91986 -3.14159]
ub: 
[3.14159 1.91986 2.96706 2.35619 3.14159 1.91986 3.14159 3.14159 1.91986
 2.96706 2.35619 3.14159 1.91986 3.14159 3.14159 1.91986 2.96706 2.35619
 3.14159 1.91986 3.14159 3.14159 1.91986 2.96706 2.35619 3.14159 1.91986
 3.14159 3.14159 1.91986 2.96706 2.35619 3.14159 1.91986 3.14159 3.

In [8]:
import open3d as o3d
# initialization of the trajectory
q_init = np.zeros((n_waypoints, len(robot.selected_joint_dofids)))
q_init = np.linspace(q_start, q_target, n_waypoints+1)[1:]  # optimization variable: starting from the second point
# visualize the initalization
vis = []
for i in range(n_waypoints):
    robot.set_selected_joint_values(q_init[i])
    vis_i = scene.visualize(False)
    vis += vis_i
o3d.visualization.draw_geometries(vis)


In [9]:
robot.set_selected_joint_values(q_init[1])
results = scene.compute_collision_min_dist_total(dist_upper_bound=0.05, security_margin=solver.safety_margin, full=True)
for col_i in range(len(results)):
    link1, link2, obj_idx1, obj_idx2, distance_result = results[col_i]
    # check the distance
    distance = distance_result['distance']
    if distance < 0.01:
        print('link1: ', link1)
        print('link2: ', link2)
        print('distance: ', distance)

In [10]:
# check if the initialization satisfies the constraints
constr = solver.constraints(q_init.flatten())
cl = solver.cl
cu = solver.cu
# check if constr >= cl
print("constr >= cl: ", np.all(constr >= cl))
# check if constr <= cu
print("constr <= cu: ", np.all(constr <= cu))
# check which constraints are violated
col_constrs = solver.collision_constraints(q_init.flatten())
print(col_constrs[col_constrs<0.01])
print('collision constraint satisfaction: ', np.all(col_constrs >= 0.01))
dist_constrs = solver.position_distance_constraints(q_init.flatten())
max_dist = dist
print('distance constraint satisfaction: ', np.all(dist_constrs <= max_dist))

constr >= cl:  True
constr <= cu:  True
[]
collision constraint satisfaction:  True
distance constraint satisfaction:  True


In [11]:
# check the jacobian
grad = solver.waypoint_distance_gradient(q_init.flatten())
print('waypoint distance gradient: ')
print(grad)

waypoint distance gradient: 
[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  1.11022302e-16
  1.38777878e-17  0.00000000e+00  1.11022302e-16  2.77555756e-17
  0.00000000e+00  1.66533454e-16 -1.11022302e-16 -4.16333634e-17
  1.11022302e-16 -1.66533454e-16 -4.16333634e-17 -1.38777878e-17
 -2.77555756e-16  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  1.11022302e-16  1.11022302e-16 -3.33066907e-16  2.77555756e-16
  6.93889390e-17  4.16333634e-17  4.99600361e-16  0.00000000e+00
 -1.24900090e-16  4.44089210e-16 -4.99600361e-16 -1.24900090e-16
 -1.11022302e-16 -4.44089210e-16 -2.22044605e-16  2.77555756e-17
 -1.11022302e-16  3.88578059e-16  9.71445147e-17  1.38777878e-16
 -1.11022302e-16  2.22044605e-16  8.32667268e-17 -3.33066907e-16
  1.11022302e-16 -6.93889390e-17 -2.77555756e-17  1.11022302e-16
 -2.22044605e-16 -1.80411242e-16  6.66133815e-16 -6.10622664e

In [12]:
solver.add_option('mu_strategy', 'adaptive')
solver.add_option('tol', 1e-1)
solver.add_option('derivative_test', 'first-order')

x, info = solver.solve(q_init.flatten())


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.16, running with linear solver MUMPS 5.7.2.

waypoint distance:  58.865728675052715
terminal pose distance:  2.6181978943251547


/home/yinglong/Documents/research/task_motion_planning/non-prehensile-manipulation/motoman_ws/src/lab_vbnpm/playgrounds/optimal_control/planning_scene.py:150: RuntimeWarning: invalid value encountered in divide
  normal = normal / np.linalg.norm(normal)


Starting derivative checker for first derivatives.

waypoint distance:  58.86572863128157
terminal pose distance:  2.6181978943251547
* jac_g [   19,    0] = -6.1314295618931643e-06 v  ~  0.0000000000000000e+00  [ 6.131e-02]
waypoint distance:  58.86572867298088
terminal pose distance:  2.6181978943251547
* jac_g [   19,    1] =  3.0754535296977159e-03 v  ~  0.0000000000000000e+00  [ 3.075e+01]
waypoint distance:  58.86572869195688
terminal pose distance:  2.6181978943251547
waypoint distance:  58.86572867324141
terminal pose distance:  2.6181978943251547
waypoint distance:  58.86572867588713
terminal pose distance:  2.6181978943251547
waypoint distance:  58.86572866555424
terminal pose distance:  2.6181978943251547
waypoint distance:  58.86572866881208
terminal pose distance:  2.6181978943251547
waypoint distance:  58.865728703949806
terminal pose distance:  2.6181978943251547
* jac_g [  345,    7] = -2.2093578004696724e-05 v  ~  0.0000000000000000e+00  [ 2.209e-01]
waypoint distance:

KeyboardInterrupt: 

In [12]:
solver.terminal_pose_objective(q_init.flatten())

0.0

In [13]:
info

{'x': array([ 0.14892663, -0.13067784,  0.04730164,  0.17229701, -0.19314771,
         0.17146462, -0.26915152,  0.32332814, -0.25706222,  0.09331056,
         0.34015886, -0.38096652,  0.33881714, -0.54606563,  0.44826995,
        -0.38526492,  0.14058463,  0.51228865, -0.57399011,  0.51005655,
        -0.82887502,  0.60271126, -0.51770308,  0.18725846,  0.68235978,
        -0.7645136 ,  0.67942012, -1.09983323,  0.75531347, -0.6438025 ,
         0.23378516,  0.85193686, -0.95467876,  0.8482696 , -1.37774235,
         0.90255557, -0.76701492,  0.28064553,  1.02260059, -1.14540691,
         1.01809491, -1.65965153,  1.05463449, -0.90465536,  0.32707927,
         1.19014135, -1.33389793,  1.18378424, -1.93618625,  1.20026096,
        -1.02752014,  0.37397081,  1.35859842, -1.52440308,  1.34927324,
        -2.20984344,  1.3469234 , -1.15519964,  0.41727296,  1.52134701,
        -1.70889247,  1.50840558, -2.47093386,  1.52525674, -1.29802867,
         0.49628572,  1.68564059, -1.89783139,